# 03b — Historical Features (rolling)

**Input:**  `data/processed/sfu_clean.db`, `data/processed/03a_train.csv`, `data/processed/03a_test.csv`
**Output:** `data/processed/03b_train.csv`, `data/processed/03b_test.csv`

### Why rolling instead of full-aggregate?

The old approach computed hist_ features from ALL train terms and attached the same
numbers to every row including early 2020 rows. This means a 2020 row could see
hist_n_fall_offerings=5 which includes falls from 2021-2024 — future data.
That caused AUC=1.0 on model_offered later (perfect scores = leakage).

**Fix:** for each grid row (course C, term T), hist_ features are computed from
offerings where course_id == C AND term came STRICTLY BEFORE T.

**Test rows** (2025) use full 2020-2024 history — no leakage there.
**Train rows** use only the terms that preceded them chronologically.

## 0. Setup

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DB = Path('../data/processed/sfu_clean.db')
OUT_PATH = Path('../data/processed')

assert CLEAN_DB.exists(),                   f'not found: {CLEAN_DB}'
assert (OUT_PATH/'03a_train.csv').exists(), 'run 03a first'
assert (OUT_PATH/'03a_test.csv').exists(),  'run 03a first'
print('ready')

ready


## 1. Load

In [2]:
conn = sqlite3.connect(CLEAN_DB)
df   = pd.read_sql('SELECT * FROM offerings', conn)
conn.close()

train_base = pd.read_csv(OUT_PATH / '03a_train.csv')
test_base  = pd.read_csv(OUT_PATH / '03a_test.csv')

print(f'offerings:   {len(df):,} rows')
print(f'train_base:  {len(train_base):,} rows')
print(f'test_base:   {len(test_base):,} rows')

offerings:   33,168 rows
train_base:  47,760 rows
test_base:   9,552 rows


## 2. Define train boundary and build term sequence

In [3]:
TRAIN_YEAR = 2024
TRAIN_TERM = 3

train_mask = (
    (df['year'] < TRAIN_YEAR) |
    ((df['year'] == TRAIN_YEAR) & (df['term_order'] <= TRAIN_TERM))
)
train_raw = df[train_mask].copy()
train_raw['fill_rate'] = train_raw['enrolled'] / train_raw['capacity']

# sequential index for ALL terms (train + test) — used for grid joining
all_terms = (
    df[['ml_term_id','year','term','term_order']]
    .drop_duplicates()
    .sort_values(['year','term_order'])
    .reset_index(drop=True)
)
all_terms['seq_idx'] = range(len(all_terms))

# same index but train-only — needed for n_terms_since computation
train_terms = all_terms[all_terms['ml_term_id'].isin(train_raw['ml_term_id'].unique())].copy()

print(f'train_raw: {len(train_raw):,} rows  |  unique courses: {train_raw["ml_course_id"].nunique():,}')
print(f'all_terms: {len(all_terms)} terms  |  train_terms: {len(train_terms)} terms')
print()
print('term sequence:')
print(all_terms[['year','term','seq_idx']].to_string(index=False))

train_raw: 27,327 rows  |  unique courses: 3,055
all_terms: 18 terms  |  train_terms: 15 terms

term sequence:
 year   term  seq_idx
 2020 spring        0
 2020 summer        1
 2020   fall        2
 2021 spring        3
 2021 summer        4
 2021   fall        5
 2022 spring        6
 2022 summer        7
 2022   fall        8
 2023 spring        9
 2023 summer       10
 2023   fall       11
 2024 spring       12
 2024 summer       13
 2024   fall       14
 2025 spring       15
 2025 summer       16
 2025   fall       17


## 3. Build rolling hist features for TRAIN rows

Strategy:
1. Aggregate offerings to course × term level (sum sections within same term)
2. Sort by [course, seq_idx] and compute cumulative sums
3. Shift by 1 — this gives "cumulative history before this term"
4. Expand to all course × train-term combinations and forward-fill
   (if course didn't run in term T, its hist at T = hist from its last run)

In [4]:
# add seq_idx to train offerings
train_seq = train_raw.merge(all_terms[['ml_term_id','seq_idx']], on='ml_term_id', how='left')

# aggregate to course × term level
ct = (
    train_seq
    .groupby(['ml_course_id','seq_idx','term'])
    .agg(
        n_sec = ('offering_id', 'count'),
        cap   = ('capacity',    'sum'),
        enr   = ('enrolled',    'sum'),
    )
    .reset_index()
    .sort_values(['ml_course_id','seq_idx'])
)

# term type flags
ct['is_spring'] = (ct['term'] == 'spring').astype(int)
ct['is_summer'] = (ct['term'] == 'summer').astype(int)
ct['is_fall']   = (ct['term'] == 'fall').astype(int)

print(f'course × term events: {len(ct):,}')
ct.head(3)

course × term events: 19,777


,ml_course_id,seq_idx,term,n_sec,cap,enr,is_spring,is_summer,is_fall
0,1,2,fall,1,120,77,0,0,1
1,1,5,fall,1,120,97,0,0,1
2,1,8,fall,1,120,54,0,0,1


In [5]:
# cumulative sums per course (includes the current term)
for col in ['n_sec','cap','enr','is_spring','is_summer','is_fall']:
    ct[f'cum_{col}'] = ct.groupby('ml_course_id')[col].cumsum()
ct['cum_n_offerings'] = ct.groupby('ml_course_id').cumcount() + 1

# also track seq_idx of most recent run (for n_terms_since)
ct['last_run_seq'] = ct['seq_idx']

# shift by 1 to get "history BEFORE this term"
# row 0 for each course gets NaN → becomes 0 (no history yet)
shift_cols = ['cum_n_offerings','cum_n_sec','cum_cap','cum_enr',
              'cum_is_spring','cum_is_summer','cum_is_fall','last_run_seq']
for col in shift_cols:
    ct[f'prev_{col}'] = ct.groupby('ml_course_id')[col].shift(1)

print('cumulative + shifted columns computed')
ct[['ml_course_id','seq_idx','cum_n_offerings','prev_cum_n_offerings','prev_last_run_seq']].head(6)

cumulative + shifted columns computed


,ml_course_id,seq_idx,cum_n_offerings,prev_cum_n_offerings,prev_last_run_seq
0,1,2,1,NaN,NaN
1,1,5,2,1.0,2.0
2,1,8,3,2.0,5.0
3,1,11,4,3.0,8.0
4,1,12,5,4.0,11.0
5,1,14,6,5.0,12.0


In [6]:
# extract the "before" snapshot per course × term
hist_at_term = ct[[
    'ml_course_id','seq_idx',
    'prev_cum_n_offerings','prev_cum_n_sec','prev_cum_cap','prev_cum_enr',
    'prev_cum_is_spring','prev_cum_is_summer','prev_cum_is_fall',
    'prev_last_run_seq'
]].rename(columns={
    'prev_cum_n_offerings':  'hist_n_offerings',
    'prev_cum_n_sec':        'hist_n_sections_total',
    'prev_cum_cap':          'hist_capacity_total',
    'prev_cum_enr':          'hist_enrolled_total',
    'prev_cum_is_spring':    'hist_n_spring_offerings',
    'prev_cum_is_summer':    'hist_n_summer_offerings',
    'prev_cum_is_fall':      'hist_n_fall_offerings',
    'prev_last_run_seq':     'last_run_seq',
})

print(f'hist_at_term: {len(hist_at_term):,} rows (only terms where course ran)')

hist_at_term: 19,777 rows (only terms where course ran)


In [7]:
# expand to ALL course × ALL train-term combinations
# for terms where course didn't run, forward-fill from the last term it ran

train_courses = train_raw['ml_course_id'].unique()
train_seq_idxs = train_terms['seq_idx'].values

full_grid = pd.DataFrame(
    [(c, s) for c in train_courses for s in train_seq_idxs],
    columns=['ml_course_id','seq_idx']
)

# merge hist_at_term into full grid
full_grid = full_grid.merge(hist_at_term, on=['ml_course_id','seq_idx'], how='left')
full_grid = full_grid.sort_values(['ml_course_id','seq_idx'])

HIST_COLS = ['hist_n_offerings','hist_n_sections_total','hist_capacity_total',
             'hist_enrolled_total','hist_n_spring_offerings',
             'hist_n_summer_offerings','hist_n_fall_offerings','last_run_seq']

# forward-fill: for terms where course didn't run, carry forward the last snapshot
full_grid[HIST_COLS] = full_grid.groupby('ml_course_id')[HIST_COLS].ffill().fillna(0)

# n_terms_since_last_offered = current seq_idx - last seq the course ran
# 0 = ran very recently, large = hasn't run in a long time
full_grid['n_terms_since_last_offered'] = (
    full_grid['seq_idx'] - full_grid['last_run_seq']
).clip(lower=0).astype(int)
# cold start (never ran): last_run_seq=0 and seq_idx>0 → may give large number
# set cold start (hist_n_offerings==0) to 99
full_grid.loc[full_grid['hist_n_offerings'] == 0, 'n_terms_since_last_offered'] = 99
full_grid = full_grid.drop(columns='last_run_seq')

print(f'full_grid: {len(full_grid):,} rows  x  {full_grid.shape[1]} cols')
full_grid.head(3)

full_grid: 45,825 rows  x  10 cols


,ml_course_id,seq_idx,hist_n_offerings,hist_n_sections_total,hist_capacity_total,hist_enrolled_total,hist_n_spring_offerings,hist_n_summer_offerings,hist_n_fall_offerings,n_terms_since_last_offered
25590,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99
25591,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99
25592,1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99


### 3a. hist_fill_rate_std (full-train, noted caveat)

In [8]:
# fill_rate_std is a secondary feature measuring consistency of fill rates.
# Rolling std per row requires storing all past fill rates which is complex.
# We use the full-train std per course — this is a mild leakage for early train rows
# but the impact is limited (std measures variability, not level).
# Documented here as a known simplification.
fill_std = (
    train_raw
    .groupby('ml_course_id')['fill_rate']
    .std()
    .fillna(0)
    .round(4)
    .reset_index(name='hist_fill_rate_std')
)
full_grid = full_grid.merge(fill_std, on='ml_course_id', how='left')
full_grid['hist_fill_rate_std'] = full_grid['hist_fill_rate_std'].fillna(0)

print(f'full_grid with fill_std: {full_grid.shape}')

full_grid with fill_std: (45825, 11)


## 4. Spot check — CMPT 225 rolling history

In [9]:
# CMPT 225 should show increasing hist_n_offerings as we move through terms
# Spring 2020 → hist_n_offerings = 0 (no history yet)
# Fall 2020   → hist_n_offerings = 1 (only Spring 2020 before it, if it ran)
# etc.

conn = sqlite3.connect(CLEAN_DB)
cmpt225_id = pd.read_sql(
    "SELECT DISTINCT ml_course_id FROM offerings WHERE dept_code='CMPT' AND course_number='225'",
    conn
)['ml_course_id'].values
conn.close()

check = (
    full_grid[full_grid['ml_course_id'].isin(cmpt225_id)]
    .merge(all_terms[['seq_idx','year','term']], on='seq_idx', how='left')
    .sort_values('seq_idx')
)
print('CMPT 225 — rolling hist_n_offerings should INCREASE over time:')
check[['year','term','hist_n_offerings','hist_capacity_total',
       'hist_n_summer_offerings','n_terms_since_last_offered']].to_string(index=False)

CMPT 225 — rolling hist_n_offerings should INCREASE over time:


' year   term  hist_n_offerings  hist_capacity_total  hist_n_summer_offerings  n_terms_since_last_offered\n 2020 spring               0.0                  0.0                      0.0                          99\n 2020 summer               1.0                328.0                      0.0                           1\n 2020   fall               2.0                678.0                      1.0                           1\n 2021 spring               3.0               1078.0                      1.0                           1\n 2021 summer               4.0               1438.0                      1.0                           1\n 2021   fall               5.0               1638.0                      2.0                           1\n 2022 spring               6.0               2038.0                      2.0                           1\n 2022 summer               7.0               2438.0                      2.0                           1\n 2022   fall               8.0               

## 5. Build hist features for TEST rows

Test rows (2025) use FULL train history — no leakage because
all train data (2020–2024) came before 2025.
This is the same approach as the old 03b, applied to test only.

In [10]:
# full train history per course (same as old 03b)
test_hist_agg = (
    train_raw
    .groupby('ml_course_id')
    .agg(
        hist_n_sections_total = ('offering_id', 'count'),
        hist_capacity_total   = ('capacity',    'sum'),
        hist_enrolled_total   = ('enrolled',    'sum'),
    )
    .reset_index()
)

# n_offerings
n_off = (
    train_raw.drop_duplicates(subset=['ml_course_id','ml_term_id'])
    .groupby('ml_course_id').size().reset_index(name='hist_n_offerings')
)

# term type counts
ttc = (
    train_raw.drop_duplicates(subset=['ml_course_id','ml_term_id'])
    .groupby(['ml_course_id','term']).size().unstack(fill_value=0).reset_index()
)
for t in ['spring','summer','fall']:
    if t not in ttc.columns: ttc[t] = 0
ttc = ttc.rename(columns={
    'spring':'hist_n_spring_offerings',
    'summer':'hist_n_summer_offerings',
    'fall':  'hist_n_fall_offerings'
})[['ml_course_id','hist_n_spring_offerings','hist_n_summer_offerings','hist_n_fall_offerings']]

# n_terms_since (from end of train)
max_train_seq = train_terms['seq_idx'].max()
last_seen = (
    train_raw
    .merge(all_terms[['ml_term_id','seq_idx']], on='ml_term_id')
    .groupby('ml_course_id')['seq_idx'].max()
    .reset_index(name='last_seq')
)
last_seen['n_terms_since_last_offered'] = (max_train_seq - last_seen['last_seq']).astype(int)
last_seen = last_seen.drop(columns='last_seq')

# merge all
test_hist = test_hist_agg.merge(n_off, on='ml_course_id', how='left')
test_hist = test_hist.merge(ttc, on='ml_course_id', how='left')
test_hist = test_hist.merge(last_seen, on='ml_course_id', how='left')
test_hist = test_hist.merge(fill_std, on='ml_course_id', how='left')
test_hist['hist_fill_rate_std'] = test_hist['hist_fill_rate_std'].fillna(0)

print(f'test_hist: {len(test_hist):,} courses')
test_hist.head(3)

test_hist: 3,055 courses


,ml_course_id,hist_n_sections_total,hist_capacity_total,hist_enrolled_total,hist_n_offerings,hist_n_spring_offerings,hist_n_summer_offerings,hist_n_fall_offerings,n_terms_since_last_offered,hist_fill_rate_std
0,1,6,655,409,6,1,0,5,0,0.2111
1,2,4,134,87,4,3,1,0,1,0.2674
2,3,25,250,47,13,5,5,3,0,0.1130


## 6. Assemble final train and test outputs

In [11]:
FINAL_HIST_COLS = [
    'hist_n_offerings','hist_n_sections_total',
    'hist_capacity_total','hist_enrolled_total',
    'hist_n_spring_offerings','hist_n_summer_offerings','hist_n_fall_offerings',
    'hist_fill_rate_std','n_terms_since_last_offered'
]

# --- TRAIN: join rolling hist via seq_idx
train_with_seq = train_base.merge(
    all_terms[['ml_term_id','seq_idx']], on='ml_term_id', how='left'
)
train_out = train_with_seq.merge(
    full_grid[['ml_course_id','seq_idx'] + FINAL_HIST_COLS],
    on=['ml_course_id','seq_idx'], how='left'
).drop(columns='seq_idx')

# cold start: courses in train grid not seen in train_raw
train_out[FINAL_HIST_COLS] = train_out[FINAL_HIST_COLS].fillna(0)
train_out.loc[train_out['hist_n_offerings']==0, 'n_terms_since_last_offered'] = 99

# --- TEST: join full-train hist
test_out = test_base.merge(test_hist, on='ml_course_id', how='left')
test_out[FINAL_HIST_COLS] = test_out[FINAL_HIST_COLS].fillna(0)
test_out.loc[test_out['hist_n_offerings']==0, 'n_terms_since_last_offered'] = 99

print(f'train_out: {train_out.shape}')
print(f'test_out:  {test_out.shape}')

train_out: (47760, 30)
test_out:  (9552, 30)


In [12]:
# verify nulls
nulls = train_out.isnull().sum()
print('train nulls:')
print(nulls[nulls>0] if nulls.any() else 'none')
print()
nulls = test_out.isnull().sum()
print('test nulls:')
print(nulls[nulls>0] if nulls.any() else 'none')

train nulls:
none

test nulls:
none


In [13]:
# cold start summary
cold_train = (train_out['hist_n_offerings'] == 0).sum()
cold_test  = (test_out['hist_n_offerings']  == 0).sum()
print(f'cold start rows — train: {cold_train:,} ({cold_train/len(train_out)*100:.1f}%)  test: {cold_test:,} ({cold_test/len(test_out)*100:.1f}%)')

cold start rows — train: 17,789 (37.2%)  test: 387 (4.1%)


In [15]:
# verify CMPT 225 in final train_out — hist should increase over time
check2 = train_out[train_out['ml_course_id'].isin(cmpt225_id)]
check2[['year','term','term_order','was_offered','hist_n_offerings',
        'hist_capacity_total','hist_n_summer_offerings',
        'n_terms_since_last_offered']].sort_values(['year','term_order'])

,year,term,term_order,was_offered,hist_n_offerings,hist_capacity_total,hist_n_summer_offerings,n_terms_since_last_offered
5910,2020,spring,1,1,0.0,0.0,0.0,99.0
5911,2020,summer,2,1,1.0,328.0,0.0,1.0
5912,2020,fall,3,1,2.0,678.0,1.0,1.0
5913,2021,spring,1,1,3.0,1078.0,1.0,1.0
5914,2021,summer,2,1,4.0,1438.0,1.0,1.0
5915,2021,fall,3,1,5.0,1638.0,2.0,1.0
5916,2022,spring,1,1,6.0,2038.0,2.0,1.0
5917,2022,summer,2,1,7.0,2438.0,2.0,1.0
5918,2022,fall,3,1,8.0,2838.0,3.0,1.0
5919,2023,spring,1,1,9.0,3158.0,3.0,1.0


## 7. Save

In [16]:
train_out.to_csv(OUT_PATH / '03b_train.csv', index=False)
test_out.to_csv( OUT_PATH / '03b_test.csv',  index=False)

print(f'saved 03b_train.csv: {len(train_out):,} rows  x  {train_out.shape[1]} cols')
print(f'saved 03b_test.csv:  {len(test_out):,} rows   x  {test_out.shape[1]} cols')
print()
print('columns:', list(train_out.columns))

saved 03b_train.csv: 47,760 rows  x  30 cols
saved 03b_test.csv:  9,552 rows   x  30 cols

columns: ['ml_course_id', 'ml_term_id', 'was_offered', 'year', 'term', 'term_order', 'semester_code', 'is_covid_affected', 'course_level', 'prereq_count', 'has_coreqs', 'units', 'is_grad', 'has_designation', 'is_online', 'is_evening', 'is_burnaby', 'is_surrey', 'is_harbour_ctr', 'is_other_van', 'is_off_campus', 'hist_n_offerings', 'hist_n_sections_total', 'hist_capacity_total', 'hist_enrolled_total', 'hist_n_spring_offerings', 'hist_n_summer_offerings', 'hist_n_fall_offerings', 'hist_fill_rate_std', 'n_terms_since_last_offered']


In [17]:
tr = pd.read_csv(OUT_PATH / '03b_train.csv')
te = pd.read_csv(OUT_PATH / '03b_test.csv')
assert tr.shape == train_out.shape
assert te.shape == test_out.shape
print('verified')

verified
